![tracker](https://us-central1-vertex-ai-mlops-369716.cloudfunctions.net/pixel-tracking?path=statmike%2Fvertex-ai-mlops%2FMLOps%2FDevelopment+Environments%2FNotebooks&file=Colab+Enterprise+-+Notebook+Management.ipynb)
<!--- header table --->
<table align="left">
<tr>
  <td style="text-align: center">
    <a href="https://github.com/statmike/vertex-ai-mlops/blob/main/MLOps/Development%20Environments/Notebooks/Colab%20Enterprise%20-%20Notebook%20Management.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo">
      <br>View on<br>GitHub
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/statmike/vertex-ai-mlops/blob/main/MLOps/Development%20Environments/Notebooks/Colab%20Enterprise%20-%20Notebook%20Management.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo">
      <br>Run in<br>Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fstatmike%2Fvertex-ai-mlops%2Fmain%2FMLOps%2FDevelopment%20Environments%2FNotebooks%2FColab%20Enterprise%20-%20Notebook%20Management.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo">
      <br>Run in<br>Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/statmike/vertex-ai-mlops/main/MLOps/Development%20Environments/Notebooks/Colab%20Enterprise%20-%20Notebook%20Management.ipynb">
      <img width="32px" src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo">
      <br>Open in<br>Vertex AI Workbench
    </a>
  </td>
</tr>
<tr>
  <td colspan="4" style="text-align: right">
    <a href="https://raw.githubusercontent.com/statmike/vertex-ai-mlops/main/MLOps/Development%20Environments/Notebooks/Colab%20Enterprise%20-%20Notebook%20Management.ipynb"><img src="https://www.svgrepo.com/download/5445/download-button.svg" alt="Download icon" width="20px"></a> <a href="https://raw.githubusercontent.com/statmike/vertex-ai-mlops/main/MLOps/Development%20Environments/Notebooks/Colab%20Enterprise%20-%20Notebook%20Management.ipynb">Download Notebook</a> <i>(right-click and "Save As")</i>
  </td>
</tr>
</table>

# Colab Enterprise - Notebook Management

[Colab Enterprise](https://cloud.google.com/colab/docs/introduction) is Google Cloud's managed notebook environment for teams. It provides collaborative Jupyter notebooks with enterprise security, IAM integration, and managed runtimes.

Under the hood, Colab Enterprise stores notebooks as [Dataform](https://cloud.google.com/dataform/docs/overview) repositories — each notebook is a repository containing a `content.ipynb` file. This means the **Dataform API** can be used to list and download notebooks programmatically. For execution, the **Vertex AI NotebookExecutionJob API** submits notebooks to run on managed runtimes and writes results to GCS.

This notebook demonstrates how to manage Colab Enterprise notebooks as code:

| Operation | API | Description |
|---|---|---|
| **List** notebooks | Dataform API | Discover all notebooks in a project/region |
| **Download** a notebook | Dataform API | Fetch notebook content (base64-encoded) |
| **Modify** a notebook | Local JSON | Edit cells programmatically |
| **Execute** a notebook | Vertex AI NotebookExecutionJob | Run on a managed Colab Enterprise runtime |
| **Retrieve** results | GCS | Download the executed notebook with outputs |

**Use cases:** automated testing of notebook content, CI/CD for notebook-based workflows, batch execution of parameterized notebooks, and notebook migration between environments.

**References:**
- [Colab Enterprise documentation](https://cloud.google.com/colab/docs/introduction)
- [Dataform API reference](https://cloud.google.com/dataform/reference/rest)
- [NotebookExecutionJob API](https://cloud.google.com/vertex-ai/docs/reference/rest/v1beta1/projects.locations.notebookExecutionJobs)

---
## Colab Setup

To run this notebook in Colab run the cells in this section.  Otherwise, skip this section.

This cell will authenticate to GCP (follow prompts in the popup).

In [1]:
PROJECT_ID = 'statmike-mlops-349915' # replace with project ID

In [2]:
try:
    from google.colab import auth
    auth.authenticate_user()
    !gcloud config set project {PROJECT_ID}
    print('Colab authorized to GCP')
except Exception:
    print('Not a Colab Environment')
    pass

Not a Colab Environment


---
## Installs

The list `packages` contains tuples of package import names and install names.  If the import name is not found then the install name is used to install quietly for the current user.

In [3]:
# tuples of (import name, install name, min_version)
packages = [
    ('google.cloud.storage', 'google-cloud-storage', '2.14.0'),
    ('requests', 'requests'),
]

import importlib
install = False
for package in packages:
    if not importlib.util.find_spec(package[0]):
        print(f'installing package {package[1]}')
        install = True
        !pip install {package[1]} -U -q --user
    elif len(package) == 3:
        if importlib.metadata.version(package[0]) < package[2]:
            print(f'updating package {package[1]}')
            install = True
            !pip install {package[1]} -U -q --user

ModuleNotFoundError: No module named 'google'

### API Enablement

In [ ]:
!gcloud services enable dataform.googleapis.com aiplatform.googleapis.com storage.googleapis.com

### Restart Kernel (If Installs Occurred)

After a kernel restart the code submission can start with the next cell after this one.

In [ ]:
if install:
    import IPython
    app = IPython.Application.instance()
    app.kernel.do_shutdown(True)

---
## Setup

Inputs

In [ ]:
project = !gcloud config get-value project
PROJECT_ID = project[0]
PROJECT_ID

In [ ]:
REGION = 'us-central1'

Packages

In [ ]:
import json
import base64
import os
import time
from datetime import datetime

import requests
from google.auth import default
from google.auth.transport.requests import Request as AuthRequest
from google.cloud import storage

Clients

In [ ]:
credentials, _ = default()
gcs_client = storage.Client(project=PROJECT_ID)

parameters:

In [ ]:
DIR = 'temp/colab-enterprise'
if not os.path.exists(DIR):
    os.makedirs(DIR)

Helpers

In [ ]:
def get_token():
    """Get a fresh access token from Application Default Credentials."""
    credentials.refresh(AuthRequest())
    return credentials.token

def api_get(url):
    """Make an authenticated GET request and return the JSON response."""
    resp = requests.get(url, headers={'Authorization': f'Bearer {get_token()}'})
    resp.raise_for_status()
    return resp.json()

def api_post(url, data):
    """Make an authenticated POST request and return the JSON response."""
    resp = requests.post(
        url,
        headers={'Authorization': f'Bearer {get_token()}', 'Content-Type': 'application/json'},
        json=data
    )
    resp.raise_for_status()
    return resp.json()

---
## List Notebooks in Colab Enterprise

Colab Enterprise stores each notebook as a [Dataform repository](https://cloud.google.com/dataform/reference/rest/v1beta1/projects.locations.repositories). The notebook content lives inside the repository as a file named `content.ipynb`.

To discover notebooks, list repositories via the Dataform API and filter for those with `.ipynb` display names. The API supports pagination for projects with many notebooks.

- [Dataform API: repositories.list](https://cloud.google.com/dataform/reference/rest/v1beta1/projects.locations.repositories/list)

In [ ]:
# List all Dataform repositories (which represent Colab Enterprise notebooks)
base_url = f'https://dataform.googleapis.com/v1beta1/projects/{PROJECT_ID}/locations/{REGION}'

repos = []
page_token = ''
while True:
    url = f'{base_url}/repositories?pageSize=1000'
    if page_token:
        url += f'&pageToken={page_token}'
    data = api_get(url)
    repos.extend(data.get('repositories', []))
    page_token = data.get('nextPageToken', '')
    if not page_token:
        break

print(f'Found {len(repos)} total repositories in {REGION}')

In [ ]:
# Filter to notebook repositories (.ipynb display names)
notebooks = [
    repo for repo in repos
    if repo.get('displayName', '').endswith('.ipynb')
]

print(f'Found {len(notebooks)} notebooks:\n')
for nb in notebooks[:20]:
    print(f"  {nb.get('displayName', 'unnamed')}")
if len(notebooks) > 20:
    print(f'  ... and {len(notebooks) - 20} more')

---
## Download a Notebook

Each notebook repository contains its content as a file named `content.ipynb`. The Dataform API's `readFile` endpoint returns the file content as a base64-encoded string.

- [Dataform API: repositories.readFile](https://cloud.google.com/dataform/reference/rest/v1beta1/projects.locations.repositories/readFile)

In [ ]:
# Pick the first notebook to download (or set a specific name)
if notebooks:
    target_repo = notebooks[0]
    repo_name = target_repo['name']
    display_name = target_repo.get('displayName', 'notebook.ipynb')
    
    print(f'Downloading: {display_name}')
    print(f'Repository:  {repo_name}')
else:
    print('No notebooks found. Create a notebook in Colab Enterprise first.')
    print('Visit: https://console.cloud.google.com/vertex-ai/colab/notebooks')

In [ ]:
# Download the notebook content
if notebooks:
    file_data = api_get(
        f'https://dataform.googleapis.com/v1beta1/{repo_name}:readFile?path=content.ipynb'
    )
    
    # Decode from base64
    nb_content = base64.b64decode(file_data['contents'])
    
    # Write to local file
    local_path = os.path.join(DIR, display_name)
    with open(local_path, 'wb') as f:
        f.write(nb_content)
    
    # Parse and inspect
    nb = json.loads(nb_content)
    code_cells = [c for c in nb.get('cells', []) if c['cell_type'] == 'code']
    md_cells = [c for c in nb.get('cells', []) if c['cell_type'] == 'markdown']
    
    print(f'\nSaved to: {local_path}')
    print(f'Total cells:    {len(nb.get("cells", []))}')
    print(f'  Code cells:   {len(code_cells)}')
    print(f'  Markdown:     {len(md_cells)}')
    print(f'Kernel:         {nb.get("metadata", {}).get("kernelspec", {}).get("display_name", "unknown")}')

---
## Modify a Notebook Programmatically

Jupyter notebooks are JSON files with a well-defined structure. Each notebook contains a list of cells, where each cell has:

- `cell_type`: `"code"` or `"markdown"`
- `source`: a list of strings (one per line, each ending with `\n` except the last)
- `metadata`: cell-level metadata (optional)
- `outputs`: execution outputs (code cells only)

This makes it straightforward to modify notebooks programmatically — read the JSON, edit cells, write it back.

**Common modifications:**
- Inject setup cells (install packages, set project ID)
- Replace placeholder values with environment-specific configuration
- Add logging or timing instrumentation
- Strip outputs before uploading for execution

In [ ]:
# Read the downloaded notebook
if notebooks:
    with open(local_path) as f:
        nb = json.load(f)
    
    # Example: add a setup cell at the beginning
    setup_cell = {
        'cell_type': 'code',
        'metadata': {},
        'source': [
            '# Auto-injected setup\n',
            'import os\n',
            f"os.environ['PROJECT_ID'] = '{PROJECT_ID}'\n",
            f"print(f'Project: {PROJECT_ID}')\n",
        ],
        'outputs': [],
        'execution_count': None,
    }
    
    # Insert after the first cell (preserve any title/header)
    nb['cells'].insert(1, setup_cell)
    
    # Write the modified notebook
    modified_path = os.path.join(DIR, 'modified_' + display_name)
    with open(modified_path, 'w') as f:
        json.dump(nb, f, indent=1, ensure_ascii=False)
        f.write('\n')
    
    print(f'Modified notebook saved to: {modified_path}')
    print(f'Added setup cell at position 1')
    print(f'Total cells: {len(nb["cells"])}  (was {len(nb["cells"]) - 1})')

---
## Discover Runtime Templates

To execute a notebook in Colab Enterprise, you need a **runtime template**. Runtime templates define the compute configuration (machine type, disk, accelerators, idle timeout) and are created by administrators in the Google Cloud Console.

There are two types:
- `ONE_CLICK` — the default runtime template provided by Google Cloud
- `USER_DEFINED` — custom templates created by project administrators

- [Runtime template documentation](https://cloud.google.com/colab/docs/create-runtime-template)
- [NotebookRuntimeTemplate API](https://cloud.google.com/vertex-ai/docs/reference/rest/v1beta1/projects.locations.notebookRuntimeTemplates)

In [ ]:
# List available runtime templates
templates_url = (
    f'https://{REGION}-aiplatform.googleapis.com/v1beta1/'
    f'projects/{PROJECT_ID}/locations/{REGION}/notebookRuntimeTemplates'
)
data = api_get(templates_url)
templates = data.get('notebookRuntimeTemplates', [])

print(f'Found {len(templates)} runtime template(s):\n')
for t in templates:
    name = t.get('displayName', 'unnamed')
    rt_type = t.get('notebookRuntimeType', 'UNKNOWN')
    machine = t.get('machineSpec', {}).get('machineType', 'default')
    print(f'  {name}')
    print(f'    Type:    {rt_type}')
    print(f'    Machine: {machine}')
    print(f'    Resource: {t["name"]}')
    print()

In [ ]:
# Select a runtime template (prefer USER_DEFINED over ONE_CLICK/default)
runtime_template = None
for t in templates:
    if t.get('notebookRuntimeType') == 'USER_DEFINED':
        runtime_template = t
        break
if runtime_template is None and templates:
    runtime_template = templates[0]

if runtime_template:
    RUNTIME_TEMPLATE_NAME = runtime_template['name']
    print(f'Selected: {runtime_template.get("displayName", "unnamed")}')
    print(f'Resource: {RUNTIME_TEMPLATE_NAME}')
else:
    print('No runtime templates found.')
    print('Create one in the Console: Colab Enterprise > Runtime Templates')

---
## Execute a Notebook

The [Vertex AI NotebookExecutionJob API](https://cloud.google.com/vertex-ai/docs/reference/rest/v1beta1/projects.locations.notebookExecutionJobs) executes a notebook on a Colab Enterprise runtime and writes the executed notebook (with outputs) to GCS.

**Key concepts:**
- `directNotebookSource` — the notebook content is sent inline as base64 (no need to upload to GCS first)
- `notebookRuntimeTemplateResourceName` — which runtime template to use
- `executionUser` — runs the notebook under a specific user's credentials (required for accessing user-scoped resources like Dataproc Spark Connect)
- `serviceAccount` — alternative to `executionUser`, runs under a service account
- `gcsOutputUri` — where the executed notebook is written

**Important API notes:**
- Use the `v1beta1` endpoint. The `v1` (GA) endpoint causes `executionUser` jobs to hang in `PENDING`.
- Do NOT pass a custom `notebookExecutionJobId` with `executionUser` mode — this also causes `PENDING` hangs. Let the API generate the job ID.
- Each execution gets a fresh kernel — no state persists between notebook executions.

### Submit Execution Job

The execution user needs to have visited the Colab Enterprise OAuth consent URL at least once in the current session. Without this, jobs will hang in `PENDING` indefinitely.

For `executionUser` mode, use this consent URL (replace with your user email):

```
https://accounts.google.com/o/oauth2/v2/auth?client_id=1057398310658-ceutt0mc04ir64u0klh72a7v0tj178hn.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.profile&redirect_uri=https%3A%2F%2Fnotebooks.cloud.google.com%2Fstatic%2Foauth.html&response_type=none+gsession&access_type=offline&login_hint=YOUR_EMAIL&prompt=consent
```

In [ ]:
# Get the current user's email for executionUser mode
EXECUTION_USER = !gcloud config get-value account
EXECUTION_USER = EXECUTION_USER[0]
print(f'Execution user: {EXECUTION_USER}')

In [ ]:
# Read and encode the notebook for upload
if notebooks and runtime_template:
    with open(modified_path, 'rb') as f:
        nb_bytes = f.read()
    nb_b64 = base64.b64encode(nb_bytes).decode()
    
    # GCS output location
    GCS_OUTPUT = f'gs://{PROJECT_ID}/colab-enterprise/executions/{display_name.replace(".ipynb", "")}'
    
    # Display name for the execution job
    job_display_name = f'nb-exec-{display_name.replace(".ipynb", "").replace("_", "-")}-{int(time.time())}'
    
    print(f'Notebook:     {modified_path}')
    print(f'Size:         {len(nb_bytes):,} bytes')
    print(f'GCS output:   {GCS_OUTPUT}')
    print(f'Display name: {job_display_name}')

In [ ]:
# Submit the notebook execution job
# IMPORTANT: use v1beta1 — v1 causes executionUser jobs to hang in PENDING
if notebooks and runtime_template:
    create_url = (
        f'https://{REGION}-aiplatform.googleapis.com/v1beta1/'
        f'projects/{PROJECT_ID}/locations/{REGION}/notebookExecutionJobs'
    )
    
    exec_body = {
        'displayName': job_display_name,
        'directNotebookSource': {'content': nb_b64},
        'notebookRuntimeTemplateResourceName': RUNTIME_TEMPLATE_NAME,
        'gcsOutputUri': GCS_OUTPUT,
        'executionUser': EXECUTION_USER,
        'executionTimeout': '3600s',
    }
    
    result = api_post(create_url, exec_body)
    
    # Extract the API-generated job ID from the operation response
    op_name = result.get('name', '')
    parts = op_name.split('/')
    JOB_ID = parts[parts.index('notebookExecutionJobs') + 1]
    
    print(f'Job submitted!')
    print(f'Job ID:    {JOB_ID}')
    print(f'Operation: {op_name}')

### Poll for Completion

Notebook execution jobs are asynchronous. Poll the job status until it reaches a terminal state (`SUCCEEDED`, `FAILED`, or `CANCELLED`). For long-running notebooks, the access token is refreshed automatically on each poll.

In [ ]:
# Poll the execution job until completion
if notebooks and runtime_template:
    job_url = (
        f'https://{REGION}-aiplatform.googleapis.com/v1beta1/'
        f'projects/{PROJECT_ID}/locations/{REGION}/'
        f'notebookExecutionJobs/{JOB_ID}'
    )
    
    start_time = time.time()
    while True:
        time.sleep(30)
        job = api_get(job_url)
        state = job.get('jobState', '')
        elapsed = int(time.time() - start_time)
        
        if state == 'JOB_STATE_RUNNING':
            print(f'  Running... ({elapsed}s elapsed)')
        elif state == 'JOB_STATE_SUCCEEDED':
            print(f'  Succeeded! ({elapsed}s elapsed)')
            break
        elif state in ('JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
            error_msg = (
                job.get('error', {}).get('message')
                or job.get('status', {}).get('message')
                or state
            )
            print(f'  {state} ({elapsed}s elapsed)')
            print(f'  Error: {error_msg}')
            break
        else:
            print(f'  {state}... ({elapsed}s elapsed)')

---
## Download Executed Notebook from GCS

After successful execution, the notebook with outputs is written to GCS at:

```
{gcsOutputUri}/{jobId}/content.ipynb
```

The executed notebook contains all cell outputs, including printed text, tables, plots, and any errors that occurred during execution.

In [ ]:
# Download the executed notebook from GCS
if notebooks and runtime_template and state == 'JOB_STATE_SUCCEEDED':
    gcs_path = f'{GCS_OUTPUT}/{JOB_ID}/content.ipynb'
    
    # Parse the GCS path
    gcs_parts = gcs_path.replace('gs://', '').split('/', 1)
    bucket_name = gcs_parts[0]
    blob_path = gcs_parts[1]
    
    # Download using google-cloud-storage
    bucket = gcs_client.bucket(bucket_name)
    blob = bucket.blob(blob_path)
    
    completed_path = os.path.join(DIR, 'completed_' + display_name)
    blob.download_to_filename(completed_path)
    
    print(f'Downloaded executed notebook to: {completed_path}')
    print(f'GCS source: {gcs_path}')

---
## Review Execution Results

The executed notebook contains cell-level timing metadata and all outputs. This section parses the notebook to extract timing information and identify any errors.

In [ ]:
# Parse the executed notebook and extract timing + error info
if notebooks and runtime_template and state == 'JOB_STATE_SUCCEEDED':
    with open(completed_path) as f:
        executed_nb = json.load(f)
    
    code_cells = [c for c in executed_nb['cells'] if c['cell_type'] == 'code']
    print(f'Executed notebook: {len(executed_nb["cells"])} cells ({len(code_cells)} code)\n')
    
    # Extract timing from cell metadata
    timings = []
    errors = []
    for i, cell in enumerate(executed_nb['cells']):
        if cell['cell_type'] != 'code':
            continue
        
        # Check for errors
        for out in cell.get('outputs', []):
            if out.get('output_type') == 'error':
                errors.append({
                    'cell': i,
                    'ename': out.get('ename', ''),
                    'evalue': out.get('evalue', ''),
                })
        
        # Extract timing
        ex = cell.get('metadata', {}).get('execution', {})
        start = ex.get('iopub.execute_input') or ex.get('iopub.status.busy')
        end = ex.get('shell.execute_reply') or ex.get('iopub.status.idle')
        if start and end:
            try:
                t0 = datetime.fromisoformat(start.replace('Z', '+00:00'))
                t1 = datetime.fromisoformat(end.replace('Z', '+00:00'))
                duration = (t1 - t0).total_seconds()
                preview = ''.join(cell.get('source', []))[:60].strip()
                timings.append({'cell': i, 'seconds': duration, 'preview': preview})
            except (ValueError, TypeError):
                pass
    
    # Display timing summary
    if timings:
        total = sum(t['seconds'] for t in timings)
        print(f'Total execution time: {total:.1f}s ({len(timings)} timed cells)\n')
        
        # Show slowest cells
        slow = sorted(timings, key=lambda t: -t['seconds'])[:5]
        print('Slowest cells:')
        for t in slow:
            m, s = divmod(int(t['seconds']), 60)
            fmt = f'{m}m {s:02d}s' if m > 0 else f'{s}s'
            print(f"  Cell {t['cell']:>3}: {fmt:>8}  — {t['preview']}")
    
    # Display errors
    if errors:
        print(f'\nErrors ({len(errors)}):')
        for e in errors:
            print(f"  Cell {e['cell']}: {e['ename']}: {e['evalue'][:200]}")
    elif timings:
        print('\nNo errors detected.')

---
## Putting It All Together

The following class consolidates all operations into reusable methods. This makes it easy to integrate Colab Enterprise notebook management into automated workflows.

In [ ]:
class ColabEnterpriseManager:
    """Manage Colab Enterprise notebooks programmatically."""
    
    def __init__(self, project_id, region='us-central1'):
        self.project_id = project_id
        self.region = region
        self.credentials, _ = default()
        self.gcs = storage.Client(project=project_id)
    
    def _token(self):
        self.credentials.refresh(AuthRequest())
        return self.credentials.token
    
    def _headers(self):
        return {'Authorization': f'Bearer {self._token()}', 'Content-Type': 'application/json'}
    
    def list_notebooks(self, name_filter=None):
        """List notebook repositories, optionally filtered by display name."""
        base = f'https://dataform.googleapis.com/v1beta1/projects/{self.project_id}/locations/{self.region}'
        repos, page_token = [], ''
        while True:
            url = f'{base}/repositories?pageSize=1000'
            if page_token:
                url += f'&pageToken={page_token}'
            resp = requests.get(url, headers=self._headers())
            resp.raise_for_status()
            data = resp.json()
            repos.extend(data.get('repositories', []))
            page_token = data.get('nextPageToken', '')
            if not page_token:
                break
        notebooks = [r for r in repos if r.get('displayName', '').endswith('.ipynb')]
        if name_filter:
            notebooks = [r for r in notebooks if name_filter in r.get('displayName', '')]
        return notebooks
    
    def download_notebook(self, repo_name, output_path):
        """Download a notebook from Colab Enterprise to a local file."""
        url = f'https://dataform.googleapis.com/v1beta1/{repo_name}:readFile?path=content.ipynb'
        resp = requests.get(url, headers=self._headers())
        resp.raise_for_status()
        content = base64.b64decode(resp.json()['contents'])
        with open(output_path, 'wb') as f:
            f.write(content)
        return json.loads(content)
    
    def get_runtime_template(self):
        """Get the best available runtime template (prefers USER_DEFINED)."""
        url = (
            f'https://{self.region}-aiplatform.googleapis.com/v1beta1/'
            f'projects/{self.project_id}/locations/{self.region}/notebookRuntimeTemplates'
        )
        resp = requests.get(url, headers=self._headers())
        resp.raise_for_status()
        templates = resp.json().get('notebookRuntimeTemplates', [])
        for t in templates:
            if t.get('notebookRuntimeType') == 'USER_DEFINED':
                return t['name']
        return templates[0]['name'] if templates else None
    
    def execute_notebook(self, notebook_path, execution_user, gcs_output_uri,
                         runtime_template=None, timeout='3600s', poll_interval=30):
        """Execute a notebook on Colab Enterprise and wait for completion."""
        if runtime_template is None:
            runtime_template = self.get_runtime_template()
        
        with open(notebook_path, 'rb') as f:
            nb_b64 = base64.b64encode(f.read()).decode()
        
        stem = os.path.basename(notebook_path).replace('.ipynb', '').replace('_', '-')
        create_url = (
            f'https://{self.region}-aiplatform.googleapis.com/v1beta1/'
            f'projects/{self.project_id}/locations/{self.region}/notebookExecutionJobs'
        )
        body = {
            'displayName': f'nb-exec-{stem}-{int(time.time())}',
            'directNotebookSource': {'content': nb_b64},
            'notebookRuntimeTemplateResourceName': runtime_template,
            'gcsOutputUri': gcs_output_uri,
            'executionUser': execution_user,
            'executionTimeout': timeout,
        }
        
        resp = requests.post(create_url, headers=self._headers(), json=body)
        resp.raise_for_status()
        op = resp.json()
        parts = op['name'].split('/')
        job_id = parts[parts.index('notebookExecutionJobs') + 1]
        
        job_url = (
            f'https://{self.region}-aiplatform.googleapis.com/v1beta1/'
            f'projects/{self.project_id}/locations/{self.region}/'
            f'notebookExecutionJobs/{job_id}'
        )
        
        print(f'Submitted: {body["displayName"]} (job {job_id})')
        while True:
            time.sleep(poll_interval)
            resp = requests.get(job_url, headers=self._headers())
            resp.raise_for_status()
            job = resp.json()
            state = job.get('jobState', '')
            if state == 'JOB_STATE_SUCCEEDED':
                print(f'Succeeded!')
                return {'status': 'SUCCEEDED', 'job_id': job_id, 'job': job}
            elif state in ('JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
                error = job.get('error', {}).get('message', state)
                print(f'{state}: {error}')
                return {'status': 'FAILED', 'job_id': job_id, 'error': error, 'job': job}
            else:
                print(f'  {state}...')
    
    def download_result(self, gcs_output_uri, job_id, output_path):
        """Download the executed notebook from GCS."""
        gcs_path = f'{gcs_output_uri}/{job_id}/content.ipynb'
        parts = gcs_path.replace('gs://', '').split('/', 1)
        blob = self.gcs.bucket(parts[0]).blob(parts[1])
        blob.download_to_filename(output_path)
        return output_path

### Example: End-to-End Workflow

Use the `ColabEnterpriseManager` class to run a complete list → download → execute → retrieve workflow in a few lines:

In [ ]:
# Example usage (uncomment to run):
#
# mgr = ColabEnterpriseManager(PROJECT_ID, REGION)
#
# # List notebooks
# notebooks = mgr.list_notebooks()
# print(f'Found {len(notebooks)} notebooks')
#
# # Download the first one
# nb = mgr.download_notebook(notebooks[0]['name'], 'temp/my_notebook.ipynb')
#
# # Execute it
# result = mgr.execute_notebook(
#     notebook_path='temp/my_notebook.ipynb',
#     execution_user=EXECUTION_USER,
#     gcs_output_uri=f'gs://{PROJECT_ID}/colab-enterprise/executions/my_notebook',
# )
#
# # Download results
# if result['status'] == 'SUCCEEDED':
#     mgr.download_result(
#         f'gs://{PROJECT_ID}/colab-enterprise/executions/my_notebook',
#         result['job_id'],
#         'temp/my_notebook_completed.ipynb'
#     )
#     print('Done! Check temp/my_notebook_completed.ipynb for results.')

---
## Key Notes and Tips

### API Version: Use `v1beta1`

The `executionUser` mode for `NotebookExecutionJob` only works reliably with the **`v1beta1`** API endpoint. Using the `v1` (GA) endpoint, jobs with `executionUser` get stuck in `PENDING` indefinitely. The same jobs succeed immediately with `v1beta1`.

### Don't Set Custom Job IDs with `executionUser`

Passing a custom `notebookExecutionJobId` query parameter also causes `executionUser` jobs to hang in `PENDING`, even on `v1beta1`. Let the API generate the job ID and extract it from the operation response.

### OAuth Consent Required

Before using `executionUser` mode, the user must visit the Colab Enterprise OAuth consent URL once per session. Without this, jobs will hang in `PENDING` indefinitely. The consent URL is:

```
https://accounts.google.com/o/oauth2/v2/auth?client_id=1057398310658-ceutt0mc04ir64u0klh72a7v0tj178hn.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.profile&redirect_uri=https%3A%2F%2Fnotebooks.cloud.google.com%2Fstatic%2Foauth.html&response_type=none+gsession&access_type=offline&login_hint=YOUR_EMAIL&prompt=consent
```

### Fresh Kernel Per Execution

Each `NotebookExecutionJob` runs in a **fresh kernel**. Local files and variables do not persist between executions. If notebooks depend on artifacts from previous notebooks, use GCS to bridge files between executions.

### `serviceAccount` vs `executionUser`

| Mode | Best for | Notes |
|---|---|---|
| `executionUser` | Interactive user context, accessing user-scoped resources (Dataproc, Spark Connect) | Requires OAuth consent; uses `v1beta1` only |
| `serviceAccount` | CI/CD, automated pipelines, non-interactive execution | Works on `v1` and `v1beta1`; no consent needed |

### References

- [Colab Enterprise documentation](https://cloud.google.com/colab/docs/introduction)
- [Dataform API: repositories](https://cloud.google.com/dataform/reference/rest/v1beta1/projects.locations.repositories)
- [NotebookExecutionJob API](https://cloud.google.com/vertex-ai/docs/reference/rest/v1beta1/projects.locations.notebookExecutionJobs)
- [Runtime templates](https://cloud.google.com/colab/docs/create-runtime-template)
- [Vertex AI SDK for Python](https://cloud.google.com/python/docs/reference/aiplatform/latest)